In [7]:
import pandas as pd
import numpy as np

In [8]:
# 1. Load the raw dataset and set the index
print("Loading raw dataset...")
df_lte = pd.read_csv('./data/Dataset_02_LTE_1800.csv')
df_lte['Timestamp'] = pd.to_datetime(df_lte['Timestamp'])
df_lte['Cell_ID'] = df_lte['Base station'] + "_Sec_" + df_lte['Sector'].astype(str)

# DO NOT fillna(0) here. A missing PM counter often means the site was down.
df_indexed = df_lte.set_index('Timestamp')

Loading raw dataset...


In [9]:
# 2. Resample to 15-minute intervals per cell
print("Resampling data (Handling duplicates & preserving gaps)...")
agg_dict = {
    '4G data volume DL': 'sum',      # Assuming MB based on standard PM counters
    '4G RB utilization': 'mean',
    '4G active users DL': 'mean',    # Now including user data
    '4G CQI rank 1': 'mean'          # Now including signal quality
}
# Resample per cell. We do NOT fillna(0) yet, so true outages remain NaNs.
df_resampled = df_indexed.groupby('Cell_ID').resample('15min').agg(agg_dict)

Resampling data (Handling duplicates & preserving gaps)...


In [10]:
# 3. Engineer temporal and autoregressive lag features
print("Engineering temporal, lag, and baseline features...")
df_resampled['hour'] = df_resampled.index.get_level_values('Timestamp').hour
df_resampled['day_of_week'] = df_resampled.index.get_level_values('Timestamp').dayofweek

df_resampled['vol_lag_1'] = df_resampled.groupby(level='Cell_ID')['4G data volume DL'].shift(1)
df_resampled['rb_lag_1'] = df_resampled.groupby(level='Cell_ID')['4G RB utilization'].shift(1)
df_resampled['users_lag_1'] = df_resampled.groupby(level='Cell_ID')['4G active users DL'].shift(1)

Engineering temporal, lag, and baseline features...


In [11]:
# 4. Create Baseline Features (Seasonal Naive: 24 hours ago = 96 steps)
df_resampled['baseline_vol_24h'] = df_resampled.groupby(level='Cell_ID')['4G data volume DL'].shift(96)
df_resampled['baseline_rb_24h'] = df_resampled.groupby(level='Cell_ID')['4G RB utilization'].shift(96)

In [12]:
# 5. Generate Multi-Step Targets (1 to 6 hours ahead)
horizons = {'1h': 4, '2h': 8, '3h': 12, '4h': 16, '5h': 20, '6h': 24}
for name, steps in horizons.items():
    df_resampled[f'vol_target_{name}'] = df_resampled.groupby(level='Cell_ID')['4G data volume DL'].shift(-steps)
    df_resampled[f'rb_target_{name}'] = df_resampled.groupby(level='Cell_ID')['4G RB utilization'].shift(-steps)

In [13]:
# 6. Clean up and save the final dataset
# Drop rows where targets or 24h lags are NaN (handles the edges of the timeline naturally)
df_final = df_resampled.dropna().reset_index()

output_file = './data/engineered_features_all.csv'
df_final.to_csv(output_file, index=False)
print(f"Saved highly-engineered dataset to: {output_file} | Shape: {df_final.shape}")

Saved highly-engineered dataset to: ./data/engineered_features_all.csv | Shape: (514434, 25)
